# OpenVLA Fine-Tuning for Lite6 Robot  
## Loading dataset from Drive

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!unzip /content/drive/MyDrive/data.zip -d /content/
!rm /content/episodes/.gitkeep

Archive:  /content/drive/MyDrive/VLA/data.zip
   creating: /content/episodes/
  inflating: /content/episodes/episode_0022.hdf5  
  inflating: /content/episodes/episode_0082.hdf5  
  inflating: /content/episodes/episode_0062.hdf5  
  inflating: /content/episodes/episode_0014.hdf5  
  inflating: /content/episodes/episode_0045.hdf5  
  inflating: /content/episodes/episode_0032.hdf5  
  inflating: /content/episodes/episode_0050.hdf5  
  inflating: /content/episodes/episode_0034.hdf5  
  inflating: /content/episodes/episode_0057.hdf5  
  inflating: /content/episodes/episode_0072.hdf5  
  inflating: /content/episodes/episode_0081.hdf5  
  inflating: /content/episodes/episode_0053.hdf5  
  inflating: /content/episodes/episode_0040.hdf5  
  inflating: /content/episodes/episode_0047.hdf5  
  inflating: /content/episodes/episode_0074.hdf5  
  inflating: /content/episodes/episode_0102.hdf5  
  inflating: /content/episodes/episode_0063.hdf5  
  inflating: /content/episodes/episode_0101.hdf5  
  in

## Installing Python3.10.  
Ensure Python3.10 is used for OpenVLA finetuning to work

In [5]:
!wget https://github.com/korakot/kora/releases/download/v0.10/py310.sh
!bash ./py310.sh -b -f -p /usr/local
!python -m ipykernel install --name "py310" --user

--2026-04-29 23:50:11--  https://github.com/korakot/kora/releases/download/v0.10/py310.sh
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/266951884/0d0623be-3dec-4820-9e7b-69a3a5a75ef7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-30T00%3A29%3A53Z&rscd=attachment%3B+filename%3Dpy310.sh&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-29T23%3A29%3A36Z&ske=2026-04-30T00%3A29%3A53Z&sks=b&skv=2018-11-09&sig=wWwuJkmu2JdhSq3XuoDyxIC5AGoWDyDLPxXgvUZtev4%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NzUxMDIxMSwibmJmIjoxNzc3NTA2NjExLCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLmNvcmUud2luZG93

In [2]:
!python -V

Python 3.10.6


Installing requirements

In [ ]:
%%bash
cat > requirements.txt << 'EOF'
torch==2.2.0
torchvision==0.17.0
transformers==4.40.1
tokenizers==0.19.1
timm==0.9.10
peft==0.11.1
accelerate
bitsandbytes
h5py
huggingface_hub==0.25.0
ipywidgets
apache-beam
mlcroissant
draccus
wandb
packaging
ninja
tensorflow==2.15.0
tensorflow-datasets==4.9.3
numpy<2
EOF

In [8]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 127.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 119.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 119.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 45.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 58.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 135.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.4/436.4 kB 60.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 102.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 32.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Clone the OpenVLA repository.  
This repositroy has the boiler plate codes for storing datasets and fine-tuning the OpenVLA model.

In [6]:
!git clone https://github.com/openvla/openvla.git

Cloning into 'openvla'...
remote: Enumerating objects: 489, done.
remote: Total 489 (delta 0), reused 0 (delta 0), pack-reused 489 (from 1)
Receiving objects: 100% (489/489), 346.98 KiB | 24.78 MiB/s, done.
Resolving deltas: 100% (194/194), done.


Make the desired file changes as well  
These files `mixtures.py`, `configs.py`, `transforms.py` and `finetune.py` scripts along with `my_robot_dataset` directory can be found in [GitHub](https://github.com/PrashCode5321/cable-routing-using-vla/tree/main).

In [ ]:
!cp configs.py /content/openvla/prismatic/vla/datasets/rlds/oxe/configs.py
!cp mixtures.py /content/openvla/prismatic/vla/datasets/rlds/oxe/mixtures.py
!cp transforms.py /content/openvla/prismatic/vla/datasets/rlds/oxe/transforms.py
!cp finetune.py /content/openvla/vla_scripts/finetune.py

Need HuggingFace to download the model artifacts

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

Saving the dataset in `tensorflow-dataset` format (in accordance with the finetuning script's instructions

In [9]:
%%bash
cd my_robot_dataset/
tfds build --data_dir ~/tensorflow_datasets
cd ..

INFO[build.py]: Loading dataset  from path: /content/my_robot_dataset/my_robot_dataset_dataset_builder.py
WARNING[file_utils.py]: Variant folder /root/tensorflow_datasets/my_robot_dataset/1.0.0 has no dataset_info.json
INFO[cli_utils.py]: download_and_prepare for dataset my_robot_dataset/1.0.0...
INFO[dataset_builder.py]: Generating dataset my_robot_dataset (/root/tensorflow_datasets/my_robot_dataset/1.0.0)
[Builder] Found 103 episodes: [PosixPath('/content/episodes/episode_0000.hdf5'), PosixPath('/content/episodes/episode_0001.hdf5'), PosixPath('/content/episodes/episode_0002.hdf5'), PosixPath('/content/episodes/episode_0003.hdf5'), PosixPath('/content/episodes/episode_0004.hdf5'), PosixPath('/content/episodes/episode_0005.hdf5'), PosixPath('/content/episodes/episode_0006.hdf5'), PosixPath('/content/episodes/episode_0007.hdf5'), PosixPath('/content/episodes/episode_0008.hdf5'), PosixPath('/content/episodes/episode_0009.hdf5'), PosixPath('/content/episodes/episode_0010.hdf5'), PosixPat

I0000 00:00:1777507141.091847   11048 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777507141.157013   11048 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777507142.757566   11048 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]
Generating train examples...: 0 examples [00:00, ? example

Install the OpenVLA prismatic as this is not available in PyPI

In [10]:
%%bash
cd /content/openvla
pip install -e .

Obtaining file:///content/openvla
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Cloning https://github.com/moojink/dlimp_openvla to /tmp/pip-install-4xoxxx8c/dlimp_e7dbd58bb8e4447397844002940d91bb
  Resolved https://github.com/moojink/dlimp_openvla to commit 040105d256bd28866cc6620621a3d5f7b6b91b46
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 99.0 MB/s 

  Running command git clone --filter=blob:none --quiet https://github.com/moojink/dlimp_openvla /tmp/pip-install-4xoxxx8c/dlimp_e7dbd58bb8e4447397844002940d91bb


In [20]:
%%bash

cd /content/openvla

WANDB_MODE=disabled torchrun --standalone --nnodes 1 --nproc-per-node 1 vla-scripts/finetune.py \
  --data_root_dir /root/tensorflow_datasets \
  --dataset_name my_robot_dataset \
  --run_root_dir /content/openvla/runs \
  --batch_size 4 \
  --grad_accumulation_steps 4 \
  --learning_rate 5e-4 \
  --max_steps 20000 \
  --shuffle_buffer_size 1000 \
  --image_aug true \
  --use_lora true \
  --lora_rank 32 \
  --save_steps 2000 \

Fine-tuning OpenVLA Model `openvla/openvla-7b` on `my_robot_dataset`
trainable params: 110,828,288 || all params: 7,652,065,472 || trainable%: 1.4483
04/30 [01:47:48] INFO     | >> [*] Loading existing dataset    data_utils.py:208
                          statistics from                                       
                          /root/tensorflow_datasets/my_robot_d                  
                          ataset/1.0.0/dataset_statistics_8be4                  
                          38b5b9fceb464b48123e5a5466249c199d24                  
                          48f1b550bdfbada405f2ccfe.json.                        

######################################################################################
# Loading the following 1 datasets (incl. sampling weight):                         #
# my_robot_dataset: ========================================================1.000000 #
######################################################################################

04/30 [01:47:49

2026-04-30 01:47:16.552221: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-30 01:47:16.603375: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-30 01:47:16.603435: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-30 01:47:16.604890: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-30 01:47:16.612226: I tensorflow/core/platform/cpu_feature_guar

In [21]:
!zip -r artifacts.zip /content/openvla/runs/openvla-7b+my_robot_dataset+b16+lr-0.0005+lora-r32+dropout-0.0--image_aug

  adding: content/openvla/runs/openvla-7b+my_robot_dataset+b16+lr-0.0005+lora-r32+dropout-0.0--image_aug/ (stored 0%)
  adding: content/openvla/runs/openvla-7b+my_robot_dataset+b16+lr-0.0005+lora-r32+dropout-0.0--image_aug/tokenizer.model (deflated 55%)
  adding: content/openvla/runs/openvla-7b+my_robot_dataset+b16+lr-0.0005+lora-r32+dropout-0.0--image_aug/tokenizer.json (deflated 74%)
  adding: content/openvla/runs/openvla-7b+my_robot_dataset+b16+lr-0.0005+lora-r32+dropout-0.0--image_aug/model-00002-of-00004.safetensors (deflated 21%)
  adding: content/openvla/runs/openvla-7b+my_robot_dataset+b16+lr-0.0005+lora-r32+dropout-0.0--image_aug/preprocessor_config.json (deflated 73%)
  adding: content/openvla/runs/openvla-7b+my_robot_dataset+b16+lr-0.0005+lora-r32+dropout-0.0--image_aug/model.safetensors.index.json (deflated 96%)
  adding: content/openvla/runs/openvla-7b+my_robot_dataset+b16+lr-0.0005+lora-r32+dropout-0.0--image_aug/special_tokens_map.json (deflated 78%)
  adding: content/op

In [27]:
from google.colab import files
files.download('/content/artifacts.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Save the artifacts in persistent storage.

In [ ]:
!cp /content/artifacts.zip "/content/drive/MyDrive/"